# Fetch popular Steam game IDs

Fetch up to 5,000 unique game IDs using SteamSpy's owner-ranked `all` endpoint (pages 0–4). This uses estimated ownership as a popularity proxy, rather than current player counts.

Run all cells to write `popular_game_ids.json` into the project data directory. Requests are spaced at least 60 seconds apart; five pages take roughly four minutes plus request time. Only Python standard-library modules are required.

API reference: https://steamspy.com/api.php


In [6]:
from pathlib import Path
import sys

# Support kernels started from the project root or the notebooks folder.
PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "src").is_dir() and (path / "notebooks").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise RuntimeError("Start the notebook from the project root or notebooks folder")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import json
import math
import time
from pathlib import Path
from urllib.error import HTTPError
from urllib.parse import urlencode
from urllib.request import Request, urlopen

GAME_LIMIT = 5000
PAGE_SIZE = 1000
REQUEST_INTERVAL = 60
DATA_DIR = PROJECT_ROOT / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_FILE = DATA_DIR / "popular_game_ids.json"


In [7]:
def fetch_page(page: int) -> dict:
    """Fetch one SteamSpy page; retry rate limits and temporary server errors."""
    url = "https://steamspy.com/api.php?" + urlencode({"request": "all", "page": page})
    request = Request(url, headers={
        "Accept": "application/json",
        "User-Agent": "GameRecommender/1.0",
    })
    for attempt in range(3):
        try:
            with urlopen(request, timeout=60) as response:
                data = json.load(response)
            break
        except HTTPError as exc:
            if attempt == 2 or (exc.code != 429 and not 500 <= exc.code < 600):
                raise
            retry_after = exc.headers.get("Retry-After", "")
            delay = max(REQUEST_INTERVAL, int(retry_after) if retry_after.isdigit() else 0)
            print(f"HTTP {exc.code}; retrying in {delay} seconds...", flush=True)
            time.sleep(delay)
    if not isinstance(data, dict) or not data:
        raise ValueError(f"Page {page}: expected a nonempty mapping of games")
    if not all(isinstance(game, dict) and "appid" in game for game in data.values()):
        raise ValueError(f"Page {page}: unexpected SteamSpy response format")
    return data


In [8]:
if not isinstance(GAME_LIMIT, int) or GAME_LIMIT <= 0:
    raise ValueError("GAME_LIMIT must be a positive integer")

app_ids = []
seen = set()
for page in range(math.ceil(GAME_LIMIT / PAGE_SIZE)):
    if page:
        print(f"Waiting {REQUEST_INTERVAL} seconds before the next page...", flush=True)
        time.sleep(REQUEST_INTERVAL)
    print(f"Fetching SteamSpy page {page}...", flush=True)
    games = fetch_page(page)
    for game in games.values():
        app_id = int(game["appid"])
        # SteamSpy uses 999999 to represent data hidden at a developer's request.
        if 0 < app_id <= 2**32 - 1 and app_id != 999999 and app_id not in seen:
            seen.add(app_id)
            app_ids.append(app_id)
    app_ids = app_ids[:GAME_LIMIT]
    OUTPUT_FILE.write_text(json.dumps(app_ids, indent=2) + "\n", encoding="utf-8")
    print(f"Saved {len(app_ids)} unique game IDs to {OUTPUT_FILE.resolve()}", flush=True)

app_ids = app_ids[:GAME_LIMIT]
if len(app_ids) < GAME_LIMIT:
    print(f"Only {len(app_ids)} usable unique IDs were returned; requested {GAME_LIMIT}.")


Fetching SteamSpy page 0...
Saved 1000 unique game IDs to C:\projects\GameRecommender\data\popular_game_ids.json
Waiting 60 seconds before the next page...


KeyboardInterrupt: 

In [ ]:
# IDs are saved after each successful page fetch.
app_ids[:10]
